# MegaDescriptor-L-384
- Run inference with MegaDescriptor-L-384 (https://huggingface.co/BVRA/MegaDescriptor-L-384)

In [1]:
# import pandas as pd
# from torchvision import transforms as T
# from timm import create_model

# from wildlife_tools.features import DeepFeatures
# from wildlife_tools.data import WildlifeDataset
# from wildlife_tools.similarity import CosineSimilarity
# from wildlife_tools.inference import KnnClassifier


# datasets = [
#     'BirdIndividualID',
#     'SealID',
#     'FriesianCattle2015',
#     'ATRW',
#     'NDD20',
#     'SMALST',
#     'SeaTurtleIDHeads',
#     'AAUZebraFish',
#     'CZoo',
#     'CTai',
#     'Giraffes',
#     'HyenaID2022',
#     'MacaqueFaces',
#     'OpenCows2020',
#     'StripeSpotter',
#     'AerialCattle2017',
#     'GiraffeZebraID',
#     'IPanda50',
#     'WhaleSharkID',
#     'FriesianCattle2017',
#     'Cows2021',
#     'LeopardID2022',
#     'NOAARightWhale',
#     'HappyWhale',
#     'HumpbackWhaleID',
#     'LionData',
#     'NyalaData',
#     'ZindiTurtleRecall',
#     'BelugaID',
#     ]

# model = create_model("hf-hub:BVRA/wildlife-mega-L-384", pretrained=True)
# extractor = DeepFeatures(model, device='cuda')

# root_images = '../data/images/size-518'
# root_metadata = '../data/metadata/datasets'

In [2]:
# This code replaces the code above to use the model that was trained in validation-Training-MegaDescriptor-L-384.ipynb
import pandas as pd
from torchvision import transforms as T
import torch

from timm import create_model

from wildlife_tools.features import DeepFeatures
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier

from welfareobs.detectron.welfareobs_dataset import WelfareObsDataset
from welfareobs.utils.padded_square_transform import PaddedSquareTransform



In [3]:
# results = {}
# for name in datasets:
#     metadata = pd.read_csv(f'{root_metadata}/{name}/metadata.csv', index_col=0)

#     transform = T.Compose([
#         T.Resize(size=(384, 384)),
#         T.ToTensor(),
#         T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ])

#     database = WildlifeDataset(
#         metadata=metadata.query('split == "train"'),
#         root=f'{root_images}/{name}/',
#         transform=transform,
#     )

#     query = WildlifeDataset(
#         metadata=metadata.query('split == "test"'),
#         root=f'{root_images}/{name}/',
#         transform=transform,
#     )

#     matcher = CosineSimilarity()
#     similarity = matcher(query=extractor(query), database=extractor(database))
#     preds = KnnClassifier(k=1, database_labels=database.labels_string)(similarity)

#     acc = sum(preds == query.labels_string) / len(preds)
#     print(name, acc)
#     results[name] = acc


# pd.Series(results).to_csv('results/MegaDescriptor-L-384.csv')

In [4]:
# This code replaces the loop above to just use the known dataset
model = create_model('swin_large_patch4_window12_384', num_classes=0, pretrained=True)
model.load_state_dict(torch.load("/project/data/md-validation/checkpoint.pth", weights_only=False, map_location=torch.device("cuda"))['model'])
extractor = DeepFeatures(model, device="cuda")

# transform = T.Compose([
#         T.Resize(size=(384, 384)),
#         T.ToTensor(),
#         T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ])

# My transform maintains aspect ratio using PaddedSquareTransform
transform = T.Compose([
    PaddedSquareTransform(fill=0, padding_mode="edge"),    
    T.Resize(
        size=(384,384),
        interpolation=T.InterpolationMode.BILINEAR,
        max_size=None,
        antialias=True
    ),
    T.ToTensor(),  # Convert a PIL Image or ndarray to tensor and scale the values 0->255 to 0.0->1.0
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # output[channel] = (input[channel] - mean[channel]) / std[channel] (this is the mapping for ImageNet RGB)
])

database = WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_train.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

query = WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_test.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

print("working...")
matcher = CosineSimilarity()
similarity = matcher(query=extractor(query), database=extractor(database))
preds = KnnClassifier(k=1, database_labels=database.labels_string)(similarity)
acc = sum(preds == query.labels_string) / len(preds)
print(f"WoD Accuracy: {acc}")


working...
WoD Accuracy: 0.9743589743589743


In [5]:
# Here we have a variation to evaluate finetuning the MegaDescriptor model
model = create_model("hf-hub:BVRA/wildlife-mega-L-384", pretrained=True)
model.load_state_dict(torch.load("/project/data/md-validation-finetuning/checkpoint.pth", weights_only=False, map_location=torch.device("cuda"))['model'])
extractor = DeepFeatures(model, device="cuda")

database = WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_train.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

query = WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_test.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

print("working...")
matcher = CosineSimilarity()
similarity = matcher(query=extractor(query), database=extractor(database))
preds = KnnClassifier(k=1, database_labels=database.labels_string)(similarity)
acc = sum(preds == query.labels_string) / len(preds)
print(f"Foundation finetuned with WoD Accuracy: {acc}")

working...
Foundation finetuned with WoD Accuracy: 0.9914529914529915
